# s0: Build City Distance Matrices
Reads `data/cities.csv`, calls the OpenRouteService distance matrix API, and writes:
- `data/adjacencyMatrixDist.csv` — driving distances in **miles**
- `data/adjacencyMatrixTravelTime.csv` — driving durations in **minutes**
- `data/city_distances.csv` — same data in long (pairwise) format

Both matrix files use city names as the row and column index, matching the format expected by `s2_probDefAndRPM.ipynb`.

## Configuration

In [7]:
import importlib.util, pathlib

_spec = importlib.util.spec_from_file_location(
    "api_keys",
    pathlib.Path("../../api_keys.py").resolve()
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
ORS_API_KEY = _mod.ORS_API_KEY

CITIES_CSV   = '../../data/cities.csv'
OUT_DIST_MAT = '../../data/adjacencyMatrixDist.csv'
OUT_TIME_MAT = '../../data/adjacencyMatrixTravelTime.csv'

KM_TO_MILES = 0.621371

## Load cities

In [8]:
import pandas as pd
import numpy as np

cities = pd.read_csv(CITIES_CSV)
display(cities)

,city,lat,long,pop,svi
0,Ada,40.768056,-83.825278,5334,0.390000
1,Alger,40.709722,-83.844167,837,0.594200
2,Bluffton,40.889444,-83.879167,3967,0.450800
3,Cairo,40.830833,-84.084444,517,0.190000
4,Caledonia,40.636389,-82.969444,560,0.327500
5,Carey,40.949444,-83.376111,3565,0.414200
6,Columbus Grove,40.920556,-84.059722,2160,0.350800
7,Continental,41.100278,-84.272500,1102,0.399200
8,Cridersville,40.664722,-84.130833,1791,0.676700
9,Delphos,40.861111,-84.350000,7117,0.562500


## Build ORS coordinate list
ORS expects `[longitude, latitude]` order.

In [9]:
# ORS distance_matrix expects [longitude, latitude]
coordinates = cities[['long', 'lat']].values.tolist()
city_names  = cities['city'].tolist()

print(f"{len(city_names)} cities loaded")
print(list(zip(city_names, coordinates))[:5])

36 cities loaded
[('Ada', [-83.825278, 40.768056]), ('Alger', [-83.844167, 40.709722]), ('Bluffton', [-83.879167, 40.889444]), ('Cairo', [-84.084444, 40.830833]), ('Caledonia', [-82.969444, 40.636389])]


## Call ORS distance matrix API

In [10]:
import openrouteservice as ORS

client = ORS.Client(key=ORS_API_KEY)

n = len(city_names)
indices = list(range(n))

response = client.distance_matrix(
    locations=coordinates,
    profile='driving-car',
    metrics=['distance', 'duration'],
    sources=indices,
    destinations=indices,
    units='km'
)

print("API call successful")

API call successful


## Build adjacency matrices

In [11]:
raw_distances = response['distances']   # km
raw_durations = response['durations']   # seconds

dist_miles   = np.array(raw_distances) * KM_TO_MILES          # km  → miles
time_minutes = np.array(raw_durations) / 60.0                  # sec → minutes

adj_dist = pd.DataFrame(dist_miles.round(2),   index=city_names, columns=city_names)
adj_time = pd.DataFrame(time_minutes.round(0).astype(int), index=city_names, columns=city_names)

adj_dist.index.name = 'city'
adj_time.index.name = 'city'

display("Distance matrix (miles):")
display(adj_dist)
display("Travel time matrix (minutes):")
display(adj_time)

'Distance matrix (miles):'

,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,Cridersville,Delphos,...,Ottawa,Ottoville,Pandora,Prospect,Saint Marys,Spencerville,Sycamore,Upper Sandusky,Wapakoneta,Waynesfield
city,,,,,,,,,,,,,,,,,,,,,
Ada,0.00,5.29,11.58,19.42,63.57,41.11,22.67,44.50,24.87,34.05,...,29.75,37.98,19.76,49.10,41.90,30.61,43.13,34.29,31.35,19.86
Alger,5.24,0.00,14.58,22.42,68.59,46.14,25.67,47.50,22.89,37.05,...,32.75,40.99,23.49,49.21,39.92,29.29,48.16,39.32,29.37,16.74
Bluffton,11.44,16.47,0.00,16.40,66.72,34.18,11.20,33.17,26.71,31.04,...,18.27,34.98,8.20,75.85,43.74,32.46,46.29,37.44,33.19,30.05
Cairo,19.53,24.56,16.92,0.00,74.81,48.96,6.87,28.25,15.25,16.37,...,13.95,20.31,12.94,83.94,32.29,22.31,54.38,45.53,21.73,21.77
Caledonia,63.65,68.67,66.64,74.48,0.00,40.61,77.73,99.56,86.04,89.11,...,84.80,93.04,74.82,21.62,103.07,95.04,38.53,28.24,92.51,56.89
Carey,40.15,45.18,34.05,48.73,39.88,0.00,44.74,56.31,59.04,63.37,...,43.07,67.31,36.42,49.00,76.07,64.79,11.83,14.04,65.52,62.39
Columbus Grove,22.75,27.78,11.20,6.87,78.04,44.86,0.00,21.41,21.34,22.46,...,7.11,17.31,6.10,87.17,38.37,28.40,57.60,48.77,27.82,27.86
Continental,44.40,49.42,33.17,28.25,99.68,56.58,21.41,0.00,38.31,21.79,...,15.87,16.09,24.59,108.81,44.68,31.67,68.12,70.40,44.79,45.12
Cridersville,24.66,22.64,26.91,15.28,87.03,58.95,21.37,38.35,0.00,27.96,...,28.45,31.89,27.63,80.38,20.43,18.50,66.59,57.75,9.87,12.64


'Travel time matrix (minutes):'

,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,Cridersville,Delphos,...,Ottawa,Ottoville,Pandora,Prospect,Saint Marys,Spencerville,Sycamore,Upper Sandusky,Wapakoneta,Waynesfield
city,,,,,,,,,,,,,,,,,,,,,
Ada,0,10,19,25,71,46,32,66,40,38,...,43,44,32,74,54,52,53,41,42,33
Alger,10,0,27,33,79,55,40,74,39,46,...,51,52,38,74,53,51,61,49,41,25
Bluffton,18,27,0,20,74,38,21,53,34,34,...,32,40,15,82,48,46,55,44,35,39
Cairo,25,33,21,0,80,54,11,45,31,18,...,22,24,22,88,45,33,61,50,33,38
Caledonia,71,79,73,79,0,47,87,120,98,92,...,98,98,86,30,112,107,49,35,99,90
Carey,47,55,37,54,48,0,57,81,67,67,...,61,73,49,57,81,80,20,19,69,73
Columbus Grove,33,41,21,11,88,58,0,34,40,27,...,11,29,11,97,54,42,70,58,42,47
Continental,66,74,53,45,121,82,34,0,70,36,...,29,28,43,129,78,55,101,91,71,77
Cridersville,39,39,34,31,99,67,40,70,0,42,...,51,48,40,106,31,36,81,69,19,29


## Save outputs

In [12]:
adj_dist.to_csv(OUT_DIST_MAT)
adj_time.to_csv(OUT_TIME_MAT)
print(f"Saved: {OUT_DIST_MAT}")
print(f"Saved: {OUT_TIME_MAT}")

Saved: ../../data/adjacencyMatrixDist.csv
Saved: ../../data/adjacencyMatrixTravelTime.csv
